# Resource estimation for quantum chemistry

Compare block-encoding resource models for tensor hypercontraction, double
factorization, and first quantization. The numbers below are estimates from
declared call graphs. The register scores show the order and operands of
the source composite calls.

Adapted from the Qualtran v0.7.0 chemistry resource tutorial.
Copyright 2023 Google LLC. Licensed under Apache-2.0.

## Second-quantized Hamiltonians

For the synthetic FeMoCo-sized tensor hypercontraction example, construct
the one-body correction from the central tensor and orbital factors. The
coefficient sample uses a fixed seed. The source contractions, 108 spin
orbitals, rank 350, rotation precision 16, and alias precision 10 determine
the model inputs.

In [ ]:
from io import BytesIO

from IPython.display import Image, SVG, display
from matplotlib import pyplot as plt

from examples.algorithms.chemistry_resource_tutorial import (
    ALIAS_BITS, ANGLE_BITS, ELECTRON_COUNTS, SCENARIOS, SPIN_ORBITALS,
    approximate_alpha_first_quantization, df_model, first_quantization_models,
    first_quantization_select_breakdown, first_quantization_select_score, modeled_ccz,
    thc_fixture, thc_models, thc_prepare_t_breakdown, thc_select_score, volume_from_rs_eta,
)
from examples.algorithms.double_factorization_resources import (
    double_factorization_block_score, double_factorization_block_signature,
)
from flagquantum.drawer.call_graph import export_modeled_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.register_score_data import register_score_data
from flagquantum.drawer.register_score_mpl import draw_register_score, draw_register_score_label

def display_figure(figure):
    with BytesIO() as buffer:
        figure.savefig(buffer, format="png", dpi=120, bbox_inches="tight")
        display(Image(data=buffer.getvalue()))
    plt.close(figure)

def display_score(score, short_names=None):
    short_names = short_names or {}

    def draw_short_label(ax, connector):
        label = connector["label"]
        text = short_names.get(label["text"], label["text"])
        draw_register_score_label(ax, {**connector, "label": {**label, "text": text}})

    figure, _ = draw_register_score(register_score_data(score), max_width=18,
                                    label_drawer=draw_short_label)
    display_figure(figure)

def pie_series(series, labels):
    figure, axes = plt.subplots(1, len(series), figsize=(2.15 * len(series), 3.05))
    for ax, (eta, values) in zip(axes, series.items(), strict=True):
        wedges, _, _ = ax.pie(values, autopct=lambda pct: f"{pct:.1f}%" if pct >= 2 else "")
        ax.set_title(f"{eta} electrons")
    figure.legend(wedges, labels, loc="lower center", ncol=len(labels))
    figure.subplots_adjust(bottom=0.19)
    display_figure(figure)

### THC block encoding

The LCU block consists of PREPARE, SELECT, and PREPARE adjoint.
For the CCZ-equivalent comparison, count each approximate controlled swap
as one controlled swap and price each controlled swap, And, and Toffoli
as one CCZ unit. The physical implementation can have a different T count.

In [ ]:
thc = thc_models(*thc_fixture())
parts = dict(zip(SCENARIOS, map(modeled_ccz, (thc.select, thc.prepare, thc.unprepare)), strict=True))
total = sum(parts.values())
print(f"Synthetic coefficient one-norm {thc.normalization:.7g}")
print(f"THC block: {total:,} modeled CCZ units; literature comparison 10,880; delta {total - 10880:+,}.")
print(parts)

The literature comparison excludes reflection. Splitting the total
shows the SELECT contribution and the gap between coherent preparation
and the modeled measurement-assisted erasure.

In [ ]:
figure, ax = plt.subplots(figsize=(6, 3.5))
ax.pie(tuple(parts.values()), labels=tuple(parts), autopct="%1.1f%%")
display_figure(figure)

### PREPARE T-equivalent classification

The next chart uses the source T-equivalent weights and cost categories.
The chart labels derive from the wedges returned by Matplotlib.

In [ ]:
ordered = sorted(thc_prepare_t_breakdown(thc.prepare).items(),
                 key=lambda item: item[1], reverse=True)
labels, values = zip(*ordered, strict=True)
figure, ax = plt.subplots(figsize=(8, 3.9))
wedges, _ = ax.pie(values)
ax.legend(wedges, [f"{label} ({value:,})" for label, value in ordered],
          loc="center left", bbox_to_anchor=(0.96, 0.5))
display_figure(figure)

### SELECT and its rotation subroutine

The register score shows shared angle workspace, the spin and address
swaps, both rotation pairs, and their controlled number phases.

In [ ]:
display_score(thc_select_score())

In [ ]:
display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(thc.rotations))))
rotation_cost = SPIN_ORBITALS * (ANGLE_BITS - 2)
print(f"Rotation {rotation_cost:,} and lookup {modeled_ccz(thc.rotations) - rotation_cost:,} modeled CCZ units.")

## Double factorization

The second FeMoCo-sized model has 360 auxiliary factors and 13,031
eigenvalues. The budget comparison subtracts the source's reflection
and phase-estimation allowance from the paper total.

In [ ]:
df = df_model()
df_signature = double_factorization_block_signature(360, SPIN_ORBITALS, 13031, ANGLE_BITS, 8)
display_score(double_factorization_block_score(df, df_signature), {
    "OuterPrepareDoubleFactorization": "Prepare",
    "OutputIndexedData": "Load",
    "DoubleFactorizationOneBody": "One body",
    "ReflectionUsingPrepare": "Reflect",
    "Adjoint": "Undo",
})

In [ ]:
overhead = (360 - 1).bit_length() + (SPIN_ORBITALS // 2 - 1).bit_length() + ALIAS_BITS + 1 + 2
print(f"Double factorization: {modeled_ccz(df):,} modeled CCZ units; literature comparison {21753 - overhead:,}.")

## First quantization

Use the cubic-cell approximation with density parameter 3.0 and
six bits per momentum axis. The approximate one-norm is the sum of
kinetic, electron–nuclear, and electron–electron terms. The density example
allows 3.5 electrons as a continuous parameter. The circuit models below
use integer particle counts.

In [ ]:
volume = volume_from_rs_eta(3.0, 3.5)
alpha = approximate_alpha_first_quantization(volume, 3.5, 6)
print(f"Illustrative cell volume {volume:.6g} bohr³; approximate one-norm {alpha:.6g}.")

In [ ]:
plane_wave_models = {eta: first_quantization_models(eta) for eta in ELECTRON_COUNTS}
counts = {eta: tuple(map(modeled_ccz, models)) for eta, models in plane_wave_models.items()}
for eta, values in counts.items():
    norm = approximate_alpha_first_quantization(volume_from_rs_eta(3.0, eta), eta, 6)
    print(f"{eta:2} electrons  alpha ≈ {norm:10.2f}  {values}")
pie_series(counts, SCENARIOS)

### First-quantized SELECT

The two-particle score exposes the temporary momentum banks and nuclear
position word. Kinetic and potential clauses act between the indexed
exchanges, then the temporary words are released.

In [ ]:
display_score(first_quantization_select_score(), {"SELECT T": "T", "SELECT UV": "UV"})

In [ ]:
select_breakdowns = {eta: first_quantization_select_breakdown(models[0])
                     for eta, models in plane_wave_models.items()}
for eta, row in select_breakdowns.items():
    print(f"{eta:2} electrons: {row}")
select_labels = tuple(select_breakdowns[ELECTRON_COUNTS[0]])
pie_series({eta: tuple(row.values()) for eta, row in select_breakdowns.items()}, select_labels)

The model assigns a growing fraction of SELECT's CCZ units to
multiplexed momentum exchange as the particle count increases.